In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from claude_agent_sdk import (
    ClaudeSDKClient,
    AssistantMessage,
    PostToolUseHookInput,
    PreToolUseHookInput,
    SystemMessage,
    TextBlock,
    ClaudeAgentOptions,
    tool,
    HookMatcher,
    create_sdk_mcp_server,
)
import os


@tool(
    name="get_weather",
    description="Gets the weather for a given city",
    input_schema={
        "city": str,
    },
)
async def get_weather(args):
    city = args.get("city")
    return {"content": [{"type": "text", "text": f"{city} is sunny"}]}


weather_server = create_sdk_mcp_server(
    name="weather_server",
    tools=[get_weather],
)


async def pre_tool_use(input_data: PreToolUseHookInput, tool_use_id: str, context: str):
    print(input_data["tool_name"], "->", input_data["tool_input"])
    return {}


async def post_tool_use(
    input_data: PostToolUseHookInput, tool_use_id: str, context: str
):
    print(input_data["tool_name"], "->", input_data["tool_response"])
    return {}


options = ClaudeAgentOptions(
    allowed_tools=["Write", "mcp__weather_server__*", "Skill"],
    mcp_servers={
        "weather_server": weather_server,
    },
    hooks={
        "PreToolUse": [HookMatcher(matcher=".*", hooks=[pre_tool_use])],
        "PostToolUse": [HookMatcher(matcher=".*", hooks=[post_tool_use])],
    },
    system_prompt={
        "type": "preset",
        "preset": "claude_code",
        "append": "And always speak like a pirate",
    },
    cwd=os.getcwd(),
    setting_sources=["project"],
    # system_prompt="Whatever",
)


async with ClaudeSDKClient(options=options) as client:

    # await client.query(
    #     prompt="/translate-korean how are you in this fine evening?"
    # )
    await client.query(
        prompt="translate to korean please: how are you in this fine evening?"
    )

    async for message in client.receive_response():
        if isinstance(message, SystemMessage):
            print(message)
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, TextBlock):
                    print(f"AssistantMessage TextBlock = {block.text}")

SystemMessage(subtype='init', data={'type': 'system', 'subtype': 'init', 'cwd': '/Users/nomadcoders/Documents/claude-agent-sdk', 'session_id': '5522d861-aab7-4e90-be90-06b6569ad4a3', 'tools': ['Task', 'AskUserQuestion', 'Bash', 'CronCreate', 'CronDelete', 'CronList', 'Edit', 'EnterPlanMode', 'EnterWorktree', 'ExitPlanMode', 'ExitWorktree', 'Glob', 'Grep', 'ListMcpResourcesTool', 'Monitor', 'NotebookEdit', 'PushNotification', 'Read', 'ReadMcpResourceTool', 'RemoteTrigger', 'ScheduleWakeup', 'Skill', 'TaskOutput', 'TaskStop', 'TodoWrite', 'WebFetch', 'WebSearch', 'Write', 'mcp__claude_ai_Gmail__authenticate', 'mcp__claude_ai_Gmail__complete_authentication', 'mcp__claude_ai_Google_Calendar__authenticate', 'mcp__claude_ai_Google_Calendar__complete_authentication', 'mcp__claude_ai_Google_Drive__authenticate', 'mcp__claude_ai_Google_Drive__complete_authentication', 'mcp__claude_ai_Repl__open-repl', 'mcp__weather_server__get_weather'], 'mcp_servers': [{'name': 'claude.ai Repl', 'status': 'con